# ♻️ Recycling Warehouse: Tile Surface Area Analysis & Batch Approval

Welcome to this project notebook! You will build an end-to-end vision pipeline that automatically **segments intact floor tiles and calculates usable surface area** to determine batch approval — a core task in recycling warehouse operations.

**The Problem**

A batch of reclaimed floor tiles arrives at the warehouse. Photos are taken. Your system must answer two questions:
1. *Where* are the intact (usable) regions? → **Segmentation** (U-Net)
2. *Is there enough usable surface area?* → **Binary Classification** (approve/reject based on 40% threshold)

**Pipeline Overview**

1. 🗂️ **Setup** — Clone repo, configure paths for Colab
2. 🔍 **Dataset** — Load 5000 image–mask pairs, visualize them
3. 🧱 **U-Net** — Train a basic segmentation model
4. 🔢 **FNN Classifier** — Classify from a pooled latent vector
5. 🧠 **CNN Classifier** — Classify from the full spatial latent map
6. ⚖️ **Comparison** — Threshold vs FNN vs CNN: table and metrics
7. 🚀 **Inference** — Walk through the full pipeline on one image
8. 🏆 **Challenge** — Improve the networks to beat the baseline!

**Business Logic:**
- Each batch contains multiple reclaimed floor tiles photographed together
- U-Net segments intact (usable) regions from damaged/unusable areas
- Batches with **>40% intact surface area** → **APPROVE** (sell to customers)
- Batches with **<40% intact surface area** → **REJECT** (send for manual inspection)

By the end you will understand why spatial information in the bottleneck matters for surface area classification, and how two architecturally different heads can exploit the same U-Net backbone in different ways.

---
## 1 🗂️ Setup

Clone the project repository and add it to the Python path so the dataset can be accessed. All model code is defined directly in this notebook — no external module imports needed.

In [ ]:
# Run this cell once on Colab to clone the repository
!git clone -b week2/warehouse-manager/demo https://github.com/eth-bmai-fs26/project.git
%cd project

Configure the Python path so that local imports work correctly from the cloned repository.

In [ ]:
import os
import sys

# Make sure the repo root is on the Python path so local imports work
PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Working directory:", os.getcwd())

Import all the libraries we will use throughout this notebook.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.ndimage import label as scipy_label
import torchvision.transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

### 1.1 Utility Functions

Before defining our models, we need a helper function:

1. **Dice Loss** — A segmentation-specific loss that directly measures overlap between the predicted mask and the ground truth. Unlike Binary Cross-Entropy (which treats every pixel independently), Dice Loss considers the entire region as a whole. This makes it especially useful when the foreground/background class balance is uneven.

   $$\text{Dice} = \frac{2 \cdot |P \cap T|}{|P| + |T|}$$

   We return $1 - \text{Dice}$ so that **lower is better** (standard loss convention).

In [ ]:
def dice_loss(pred, target, eps=1e-5):
    """
    Compute the Dice Loss between predicted and target masks.
    
    The Dice coefficient measures the overlap between two binary masks:
        Dice = 2 * |intersection| / (|pred| + |target|)
    
    We apply sigmoid to the raw predictions first (logits → probabilities),
    then compute Dice and return 1 - Dice so it acts as a loss.
    
    Args:
        pred:   Raw model output (logits), shape (B, 1, H, W)
        target: Ground-truth binary mask, shape (B, 1, H, W)
        eps:    Small constant to avoid division by zero
    
    Returns:
        Tensor of shape (B, 1): per-sample Dice loss (lower is better).
        Call .mean() to reduce to a scalar for backpropagation.
    """
    pred = torch.sigmoid(pred)
    intersection = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return 1 - (2 * intersection + eps) / (union + eps)

---
## 2 🔍 Dataset

We have **5000 floor tile batch images**, each paired with a binary segmentation mask that labels the intact (usable) regions.

The files follow this naming convention:

| Original image | Segmentation mask |
|---|---|
| `dataset_new/original/mosaic_0001.jpg` | `dataset_new/segmented/mosaic_0001-segmented.jpg` |
| `dataset_new/original/mosaic_0002.jpg` | `dataset_new/segmented/mosaic_0002-segmented.jpg` |
| … | … |
| `dataset_new/original/mosaic_5000.jpg` | `dataset_new/segmented/mosaic_5000-segmented.jpg` |

`SegmentationDataset` handles this via a `_mask_name()` helper that appends `-segmented` to the original stem. Both images are converted to **grayscale** and resized to **256×256**. Mask pixels above 0.5 are set to 1 (intact region), the rest to 0.

**Dataset composition:**
- **15% pristine batches** — nearly perfect tiles (high approval rate)
- **15% disaster batches** — heavily damaged (high rejection rate)  
- **70% standard batches** — typical mixed quality (where the 40% threshold matters most)

#### 2.1 Dataset Class

`SegmentationDataset` is a custom PyTorch `Dataset` that:
1. Scans the `original/` folder for all `.jpg` images
2. For each image, derives the mask filename by appending `-segmented` (e.g., `mosaic_0001.jpg` → `mosaic_0001-segmented.jpg`)
3. Loads both as **grayscale**, resizes to the target size (256×256)
4. **Binarises** the mask: any pixel above 0.5 becomes 1 (intact), the rest become 0 (damaged)

`LabeledSegmentationDataset` is a thin wrapper that also returns the approval label alongside each `(image, mask)` pair. This is essential after applying `random_split`, because subset indices no longer correspond to the original dataset order.

In [ ]:
class SegmentationDataset(Dataset):
    """
    PyTorch Dataset for paired (image, mask) segmentation data.
    
    Expects the following directory layout:
        root/
            original/       ← raw grayscale images
            segmented/      ← binary segmentation masks (suffix: -segmented)
    
    Each image is converted to grayscale, resized, and normalised to [0, 1].
    Masks are binarised at 0.5: pixels above → 1 (intact), below → 0 (damaged).
    """
    
    def __init__(self, root, image_size=(256, 256)):
        self.root = root
        self.image_size = image_size
        self.images = sorted(os.listdir(os.path.join(root, "original")))
        self.transform = T.Compose([
            T.Resize(image_size),
            T.ToTensor(),
        ])
    
    @staticmethod
    def _mask_name(img_name):
        """Derive the mask filename from the image filename.
        
        Example: 'mosaic_0001.jpg' → 'mosaic_0001-segmented.jpg'
        """
        stem, ext = os.path.splitext(img_name)
        return f"{stem}-segmented{ext}"
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_name = self.images[idx]
        mask_name = self._mask_name(img_name)
        
        # Load as grayscale (L mode) and apply transforms
        img = Image.open(os.path.join(self.root, "original", img_name)).convert("L")
        mask = Image.open(os.path.join(self.root, "segmented", mask_name)).convert("L")
        
        img = self.transform(img)
        mask = self.transform(mask)
        
        # Binarise the mask: intact regions → 1, damaged → 0
        mask = (mask > 0.5).float()
        
        return img, mask


class LabeledSegmentationDataset(Dataset):
    """
    Wraps a SegmentationDataset to also return the approval label.
    
    This is essential when using random_split: the Subset indices
    no longer match the original dataset order, so we bundle the
    label directly with each sample to avoid index mismatches.
    
    Returns: (image, mask, label) tuples
        - label = 1 → APPROVE (coverage ≥ 40%)
        - label = 0 → REJECT  (coverage < 40%)
    """
    
    def __init__(self, base_dataset, labels):
        self.base_dataset = base_dataset
        self.labels = labels
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, idx):
        img, mask = self.base_dataset[idx]
        label = self.labels[idx]
        return img, mask, label

#### 2.2 Load Data & Split

Load the ground-truth labels from the CSV, build the labelled dataset, and split into **train / validation / test** sets (70 / 15 / 15).

In [ ]:
DATA_ROOT = "week2/project1/dataset_new"

# ── 1. Load ground-truth labels from CSV ──────────────────────────────
ground_truth = pd.read_csv(f"{DATA_ROOT}/ground_truth.csv")
approval_labels = (ground_truth["status"] == "APPROVE").astype(int).tolist()

# ── 2. Build the labelled dataset ─────────────────────────────────────
ds_base = SegmentationDataset(DATA_ROOT, image_size=(256, 256))
ds_full = LabeledSegmentationDataset(ds_base, approval_labels)

# ── 3. Train / Validation / Test split (70 / 15 / 15) ────────────────
train_size = int(0.70 * len(ds_full))
val_size   = int(0.15 * len(ds_full))
test_size  = len(ds_full) - train_size - val_size

torch.manual_seed(42)                      # reproducibility
ds_train, ds_val, ds_test = random_split(ds_full, [train_size, val_size, test_size])

# ── 4. DataLoaders (each batch → (images, masks, labels)) ────────────
dl_train = DataLoader(ds_train, batch_size=32, shuffle=True)
dl_val   = DataLoader(ds_val,   batch_size=32, shuffle=False)
dl_test  = DataLoader(ds_test,  batch_size=32, shuffle=False)

print(f"Total dataset size: {len(ds_full):,} samples")
print(f"  Training set:     {len(ds_train):,} samples (70%)")
print(f"  Validation set:   {len(ds_val):,} samples (15%)")
print(f"  Test set:         {len(ds_test):,} samples (15%)")
print(f"\nEach batch yields: (images, masks, labels)")
print(f"  Image shape: {ds_base[0][0].shape}   (C, H, W)")
print(f"  Mask  shape: {ds_base[0][1].shape}   (C, H, W)")

#### 2.3 Visualise Samples

Let's visualise a few samples from the test set to verify the dataset was loaded correctly and the masks look reasonable.

In [ ]:
# Show label distribution
print(f"Total samples: {len(approval_labels)}")
print(f"Approved batches: {sum(approval_labels)} ({100*sum(approval_labels)/len(approval_labels):.1f}%)")
print(f"Rejected batches: {len(approval_labels)-sum(approval_labels)} ({100*(1-sum(approval_labels)/len(approval_labels)):.1f}%)")

# Visualize a sample of 8 batches from the test set
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
test_indices = list(ds_test.indices[:8])  # First 8 from test set

for col, idx in enumerate(test_indices):
    img, mask, label = ds_full[idx]
    status = "APPROVE" if label == 1 else "REJECT"
    coverage = ground_truth.iloc[idx]["coverage_pct"] * 100
    
    axes[0, col].imshow(img.squeeze(), cmap="gray")
    axes[0, col].set_title(f"mosaic_{idx+1:04d}\n{status} ({coverage:.1f}%)", fontsize=8)
    axes[0, col].axis("off")
    axes[1, col].imshow(mask.squeeze(), cmap="gray")
    axes[1, col].set_title("GT mask", fontsize=8)
    axes[1, col].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=11)
axes[1, 0].set_ylabel("Mask", fontsize=11)
plt.suptitle("Sample from test set (held-out data)", fontsize=13)
plt.tight_layout()
plt.subplots_adjust(hspace=0.3) 
plt.show()

#### What to look for

- Each mask should show **white regions** (value 1) over the intact/usable tiles and **black** (value 0) over damaged/unusable areas.
- **APPROVE** batches should have >40% white pixels (intact coverage)
- **REJECT** batches should have <40% white pixels
- These 8 samples are from the **test set** (held-out data not used for training)

---
## 3 🧱 U-Net Segmentation Model

We start with a simple encoder-decoder model. The encoder compresses the input image into a compact bottleneck representation, and the decoder reconstructs a segmentation mask from it.

**Architecture (`base_ch = 8`, input `256×256`)**

| Stage | Module | Output shape |
|---|---|---|
| Input | — | (B, 1, 256, 256) |
| Encoder block 1 | `Conv2d(1 → 8) + ReLU` | (B, 8, 256, 256) |
| Downsample 1 | `Conv2d stride 2` | (B, 8, 128, 128) |
| Encoder block 2 | `Conv2d(8 → 16) + ReLU` | (B, 16, 128, 128) |
| Downsample 2 | `Conv2d stride 2` | (B, 16, 64, 64) |
| **Bottleneck** | `Conv2d(16 → 16) + ReLU` | **(B, 16, 64, 64)** |
| Upsample 1 | `ConvTranspose2d` | (B, 8, 128, 128) |
| Decoder block 1 | `Conv2d(8 → 8) + ReLU` | (B, 8, 128, 128) |
| Upsample 2 | `ConvTranspose2d` | (B, 8, 256, 256) |
| Decoder block 2 | `Conv2d(8 → 8) + ReLU` | (B, 8, 256, 256) |
| Output | `Conv2d(8 → 1)` | (B, 1, 256, 256) |

The **bottleneck** is the key representation layer. Two methods expose it:
- `get_latent(x)` → GAP → vector **(B, 16)**
- `get_latent_map(x)` → raw spatial map **(B, 16, 64, 64)**

#### 3.1 Architecture Definition

In [ ]:
class SimpleConvBlock(nn.Module):
    """Convolution block: Conv → ReLU"""
    
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.block(x)


class SimpleUNet(nn.Module):
    """
    Simple encoder-decoder for semantic segmentation.
    
    Encoder compresses the input, bottleneck holds the latent representation,
    and decoder reconstructs the segmentation mask.
    """
    
    def __init__(self, in_ch=1, base_ch=8):
        super().__init__()
        self.latent_dim = base_ch * 2  # 16 channels at bottleneck
        
        # ── Encoder ───────────────────────────────────────────────────
        self.enc1 = SimpleConvBlock(in_ch, base_ch)
        self.down1 = nn.Conv2d(base_ch, base_ch, 4, 2, 1)
        
        self.enc2 = SimpleConvBlock(base_ch, base_ch * 2)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 2, 4, 2, 1)
        
        # ── Bottleneck ────────────────────────────────────────────────
        self.bottleneck = SimpleConvBlock(base_ch * 2, base_ch * 2)
        
        # ── Decoder (NO skip connections) ─────────────────────────────
        self.up1 = nn.ConvTranspose2d(base_ch * 2, base_ch, 4, 2, 1)
        self.dec1 = SimpleConvBlock(base_ch, base_ch)
        # ── Decoder ──────────────────────────────────────────────────
        self.up2 = nn.ConvTranspose2d(base_ch, base_ch, 4, 2, 1)
        self.dec2 = SimpleConvBlock(base_ch, base_ch)
        
        # ── Output head ──────────────────────────────────────────────
        self.out_conv = nn.Conv2d(base_ch, 1, 1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        d1 = self.down1(e1)
        e2 = self.enc2(d1)
        d2 = self.down2(e2)
        
        # Bottleneck
        b = self.bottleneck(d2)
        
        # Decoder — NO skip connections, just upsample
        u1 = self.up1(b)
        o1 = self.dec1(u1)
        u2 = self.up2(o1)
        o2 = self.dec2(u2)
        
        return self.out_conv(o2)

    def get_latent(self, x):
        """Returns pooled latent vector (B, 16) via Global Average Pooling."""
        e1 = self.enc1(x)  
        d1 = self.down1(e1)

        e2 = self.enc2(d1) 
        d2 = self.down2(e2)

        b = self.bottleneck(d2)
        return b.mean(dim=(2, 3))

    def get_latent_map(self, x):
        """Returns spatial bottleneck map (B, 16, 64, 64) without pooling."""
        e1 = self.enc1(x)
        d1 = self.down1(e1)

        e2 = self.enc2(d1)
        d2 = self.down2(e2)
        
        return self.bottleneck(d2)

Instantiate the model and verify the architecture summary.

In [ ]:
unet = SimpleUNet(in_ch=1, base_ch=8)
print(unet)
print(f"\nBottleneck channels (latent_dim): {unet.latent_dim}")

#### 3.2 Training the U-Net

We train the segmentation model using **Binary Cross-Entropy (BCE)** loss with **SGD** as the optimiser.

> **Note:** The `label` from each batch is ignored during U-Net training — segmentation only needs `(image, mask)` pairs. The label will be used later for classifier training.

In [ ]:
from xml.parsers.expat import model


def train_unet(model, dataloader, epochs=10, lr=1e-3):
    """
    Train the U-Net segmentation model.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    
    bce = nn.BCEWithLogitsLoss()
    # optimizer = torch.optim.SGD(model.parameters(), lr=lr) THIS GIVES 50% ACCURACY
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # was SGD lr=1e-2 THIS GIVES 90%
    
    for ep in range(epochs):
        model.train()
        epoch_loss = 0.0
        
        for img, mask, _label in dataloader:
            img  = img.to(device)
            mask = mask.to(device)
            
            pred = model(img)
            loss = bce(pred, mask)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        
        avg = epoch_loss / len(dataloader)
        print(f"[U-Net] Epoch {ep:3d}/{epochs} | Loss: {avg:.4f}")

# Train the U-Net
train_unet(unet, dl_train, epochs=10, lr=1e-2)

#### What to look for

- Loss should decrease over training epochs
- Examine the predicted masks below — how well does the model capture the tile boundaries?
- Pay attention to the quality of the segmentation — are the masks sharp or blurry? Are regions correctly identified?

In [ ]:
# Visualize U-Net predictions on TEST SET samples (unseen during training)
unet.eval()
unet_device = next(unet.parameters()).device

test_sample_indices = list(ds_test.indices[:8])  # First 8 from test set

fig, axes = plt.subplots(3, 8, figsize=(18, 9))
for col, idx in enumerate(test_sample_indices):
    img, mask, label = ds_full[idx]
    img_t = img.unsqueeze(0).to(unet_device)

    with torch.no_grad():
        pred = torch.sigmoid(unet(img_t)).squeeze().cpu()
    pred_bin = (pred > 0.5).float()
    
    status = "APPROVE" if label == 1 else "REJECT"
    coverage = ground_truth.iloc[idx]["coverage_pct"] * 100

    axes[0, col].imshow(img.squeeze(), cmap="gray")
    axes[0, col].set_title(f"mosaic_{idx+1:04d}\n{status}", fontsize=8)
    axes[0, col].axis("off")

    axes[1, col].imshow(mask.squeeze(), cmap="gray")
    axes[1, col].set_title(f"GT ({coverage:.1f}%)", fontsize=8)
    axes[1, col].axis("off")

    axes[2, col].imshow(pred_bin, cmap="gray")
    axes[2, col].set_title("U-Net pred", fontsize=8)
    axes[2, col].axis("off")

for row, lbl in enumerate(["Original", "GT mask", "U-Net pred"]):
    axes[row, 0].set_ylabel(lbl, fontsize=11)

plt.suptitle("U-Net segmentation results on TEST SET (unseen data) — BASIC MODEL", fontsize=13)
plt.tight_layout()
plt.subplots_adjust(hspace=0.3) 
plt.show()

---
## 4 🔢 FNN Classifier — Pooled Latent Vector

The first classification approach collapses the bottleneck into a single vector via **Global Average Pooling** and feeds it to a feedforward network.

```
Bottleneck map (B, 16, 64, 64)
        ↓  Global Average Pooling  [inside SimpleUNet.get_latent()]
   Latent vector (B, 16)
        ↓  Linear(16 → 1) + Sigmoid
   Approval probability p ∈ [0, 1]
```

**Architecture:**

| Layer | Input shape | Output shape |
|---|---|---|
| `Linear(16 → 1)` + Sigmoid | (B, 16) | (B,) → probability in [0, 1] |

#### 4.1 Architecture & Training

In [ ]:
class SimpleApprovalFNN(nn.Module):
    """
    Feedforward classifier on the pooled latent vector.
    Maps the latent representation to an approval probability.
    """
    
    def __init__(self, latent_dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, z):
        return self.net(z).squeeze(-1)   # (B,) probabilities

Define the training loop for the FNN classifier. The U-Net backbone is frozen — only the classifier weights are updated.

A frozen model or layer has its parameters (weights and biases) locked, preventing them from updating during training. This technique, common in transfer learning, allows us to retain valuable features learned on large datasets while only fine-tuning specific layers to adapt to a new task.

Why We Freeze Models:
- **Prevent Overfitting**: By keeping early layers locked, you avoid overfitting, particularly when training on a small dataset.

- **Save Computation**: Freezing skips backpropagation for certain layers, significantly reducing training time and memory usage.

- **Preserve Learned Features**: Pre-trained models already understand basic patterns (e.g., edges, textures), which are useful across many tasks. 

This is done by using `torch.no_grad()` to disable the gradient tape and `unet.eval()` to lock layer-specific behaviors.


In [ ]:
def train_classifier_fnn(unet, classifier, dataloader, epochs=10, lr=1e-3):
    """
    Train the FNN approval classifier.
    The U-Net is frozen: we only train the classifier head.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    unet.eval()
    classifier.to(device)
    
    criterion = nn.BCELoss()
    optimizer = torch.optim.SGD(classifier.parameters(), lr=lr)
    
    for ep in range(epochs):
        total_loss = 0.0
        for imgs, _masks, labels in dataloader:
            imgs   = imgs.to(device)
            labels = labels.float().to(device)
            
            with torch.no_grad():
                z = unet.get_latent(imgs)
            
            pred = classifier(z)
            loss = criterion(pred, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg = total_loss / len(dataloader)
        print(f"[FNN] Epoch {ep:3d}/{epochs} | Loss: {avg:.4f}")

Instantiate the FNN classifier and train it on the latent vectors extracted from the frozen U-Net.

In [ ]:
classifier_fnn = SimpleApprovalFNN(latent_dim=unet.latent_dim)
train_classifier_fnn(unet, classifier_fnn, dl_train, epochs=10, lr=1e-2)

---
## 5 🧠 CNN Classifier — Spatial Latent Map

The second approach skips the early pooling step. Instead of a 16-dimensional vector, the classifier receives the **full spatial feature map** `(B, 16, 64, 64)` and processes it with a convolutional layer before aggregation.

**Architecture:**

| Layer | Input shape | Output shape | Purpose |
|---|---|---|---|
| `Conv2d(16→4, 3×3, pad=1)` + ReLU | (B,16,64,64) | (B,4,64,64) | Spatial processing |
| `AdaptiveAvgPool2d(1)` | (B,4,64,64) | (B,4,1,1) | Global aggregate |
| `Flatten` → `Linear(4→1)` → `Sigmoid` | (B,4) | (B,) | Probability |

#### 5.1 Architecture & Training

In [ ]:
class SimpleApprovalCNN(nn.Module):
    """
    Convolutional classifier on the spatial latent map.
    Processes the bottleneck feature map before aggregation.
    """
    
    def __init__(self, in_channels=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 4, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(4, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, z_map):
        x = self.features(z_map)
        return self.classifier(x).squeeze(-1)

Define the training loop for the CNN classifier. Again, the U-Net is frozen — we only optimise the CNN head.

In [ ]:
def train_classifier_cnn(unet, classifier, dataloader, epochs=10, lr=1e-3):
    """
    Train the CNN approval classifier.
    The U-Net is frozen: we only train the classifier head.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    unet.eval()
    classifier.to(device)
    
    criterion = nn.BCELoss()
    optimizer = torch.optim.SGD(classifier.parameters(), lr=lr)
    
    for ep in range(epochs):
        total_loss = 0.0
        for imgs, _masks, labels in dataloader:
            imgs   = imgs.to(device)
            labels = labels.float().to(device)
            
            with torch.no_grad():
                z_map = unet.get_latent_map(imgs)
            
            pred = classifier(z_map)
            loss = criterion(pred, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg = total_loss / len(dataloader)
        print(f"[CNN] Epoch {ep:3d}/{epochs} | Loss: {avg:.4f}")

Instantiate the CNN classifier and train it on the spatial latent maps from the frozen U-Net.

In [ ]:
classifier_cnn = SimpleApprovalCNN(in_channels=unet.latent_dim)
train_classifier_cnn(unet, classifier_cnn, dl_train, epochs=10, lr=1e-2)

---
## 6 ⚖️ Comparison: FNN vs CNN Classifier

We now evaluate both classifiers on the **TEST SET** (held-out data) and compare their predictions against the ground truth. We also include a **direct threshold** baseline that calculates coverage from the U-Net segmentation mask (if >40% white pixels → APPROVE).

In [ ]:
def evaluate_models(unet, classifier_fnn, classifier_cnn, ds_full, ds_test, ground_truth, model_label="MODELS"):
    """
    Evaluate all three methods (threshold, FNN, CNN) on the test set.
    Returns the results DataFrame and accuracy values.
    """
    unet_device = next(unet.parameters()).device
    unet.eval()
    classifier_fnn.eval()
    classifier_cnn.eval()

    rows = []
    for idx in ds_test.indices:
        img, mask, label = ds_full[idx]
        img_t = img.unsqueeze(0).to(unet_device)

        with torch.no_grad():
            logits = unet(img_t)
            seg_mask = torch.sigmoid(logits)
            coverage = seg_mask.mean().item()
            threshold_pred = "APPROVE" if coverage > 0.4 else "REJECT"

            z = unet.get_latent(img_t)
            fnn_prob = classifier_fnn(z).item()
            fnn_pred = "APPROVE" if fnn_prob > 0.5 else "REJECT"

            z_map = unet.get_latent_map(img_t)
            cnn_prob = classifier_cnn(z_map).item()
            cnn_pred = "APPROVE" if cnn_prob > 0.5 else "REJECT"

        actual = "APPROVE" if label == 1 else "REJECT"
        actual_coverage = ground_truth.iloc[idx]["coverage_pct"]
        
        rows.append({
            "Image":       f"mosaic_{idx+1:04d}",
            "Coverage":    f"{actual_coverage*100:.1f}%",
            "Actual":      actual,
            "Threshold":   threshold_pred,
            "FNN pred":    fnn_pred,
            "CNN pred":    cnn_pred,
        })

    df = pd.DataFrame(rows)
    
    threshold_correct = sum(df["Actual"] == df["Threshold"])
    fnn_correct = sum(df["Actual"] == df["FNN pred"])
    cnn_correct = sum(df["Actual"] == df["CNN pred"])
    
    accuracies = [
        threshold_correct / len(df) * 100,
        fnn_correct / len(df) * 100,
        cnn_correct / len(df) * 100
    ]
    
    print(f"\n📊 {model_label} — Evaluating on {len(df)} test samples (unseen during training)")
    print(f"{'='*60}")
    print(f"  Threshold (40% coverage): {accuracies[0]:.1f}% ({threshold_correct}/{len(df)} correct)")
    print(f"  FNN classifier:           {accuracies[1]:.1f}% ({fnn_correct}/{len(df)} correct)")
    print(f"  CNN classifier:           {accuracies[2]:.1f}% ({cnn_correct}/{len(df)} correct)")
    print(f"{'='*60}")
    
    return df, accuracies

Next, define a plotting function to visualise the accuracy comparison as a bar chart.

In [ ]:
def plot_accuracy_comparison(accuracies, labels=None, title="Accuracy Comparison — Baseline Models"):
    """
    Bar chart comparing the accuracies of threshold, FNN, and CNN.
    """
    if labels is None:
        labels = ["Threshold\n(40% coverage)", "FNN\nClassifier", "CNN\nClassifier"]
    colors = ["#3498db", "#e67e22", "#2ecc71"]
    
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(labels, accuracies, color=colors, edgecolor='white', linewidth=1.5)
    
    for bar, acc in zip(bars, accuracies):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{acc:.1f}%", ha='center', va='bottom', fontsize=14, fontweight='bold')
    
    ax.set_ylim(0, 105)
    ax.set_ylabel("Accuracy (%)", fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()

Run the evaluation on the test set and display the first 20 predictions.

In [ ]:
df_basic, acc_basic = evaluate_models(
    unet, classifier_fnn, classifier_cnn,
    ds_full, ds_test, ground_truth,
    model_label="BASIC MODELS"
)

print("\nFirst 20 predictions:")
df_basic.head(20).set_index("Image")

Plot the accuracy comparison for the basic models.

In [ ]:
plot_accuracy_comparison(acc_basic, title="Accuracy Comparison — Basic Models")

#### Interpreting the comparison

| Method | What it sees | Inductive bias |
|---|---|---|
| **Threshold (40%)** | Predicted segmentation mask | Rule-based: count white pixels, apply 40% cutoff |
| **FNN classifier** | Pooled latent vector | Learns from a compressed summary of the bottleneck |
| **CNN classifier** | Spatial latent map | Processes spatial features before aggregating |

Take a close look at the results. Are you satisfied with the accuracy? In Section 8, you will have the opportunity to improve these models and push the accuracy significantly higher.

---
## 7 🚀 Full Pipeline Inference on a Single Image

Let us walk through the entire pipeline on a sample batch from the **test set** near the 40% threshold — the most challenging case where the decision matters most.

In [ ]:
# Find a batch near the 40% threshold from the TEST SET for demonstration
threshold_test_samples = []
for idx in ds_test.indices:
    cov = ground_truth.iloc[idx]["coverage_pct"]
    if 0.35 <= cov <= 0.45:
        threshold_test_samples.append((idx, cov))

# Pick one near the middle
IDX = threshold_test_samples[len(threshold_test_samples)//2][0] if threshold_test_samples else ds_test.indices[0]

img, gt_mask, label = ds_full[IDX]
img_t = img.unsqueeze(0).to(unet_device)

unet.eval()
classifier_fnn.eval()
classifier_cnn.eval()

with torch.no_grad():
    logits = unet(img_t)
    seg_pred = torch.sigmoid(logits)
    seg_bin = (seg_pred > 0.5).float()
    coverage = seg_pred.mean().item()
    threshold_decision = "APPROVE" if coverage > 0.4 else "REJECT"

    z = unet.get_latent(img_t)
    fnn_prob = classifier_fnn(z).item()
    fnn_decision = "APPROVE" if fnn_prob > 0.5 else "REJECT"

    z_map = unet.get_latent_map(img_t)
    cnn_prob = classifier_cnn(z_map).item()
    cnn_decision = "APPROVE" if cnn_prob > 0.5 else "REJECT"

actual_decision = "APPROVE" if label == 1 else "REJECT"
actual_coverage = ground_truth.iloc[IDX]["coverage_pct"]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(img.squeeze(), cmap="gray")
axes[0].set_title(f"Original: mosaic_{IDX+1:04d}", fontsize=11)
axes[1].imshow(gt_mask.squeeze(), cmap="gray")
axes[1].set_title(f"Ground truth mask\n{actual_coverage*100:.1f}% intact", fontsize=11)
axes[2].imshow(seg_bin.squeeze().cpu(), cmap="gray")
axes[2].set_title(f"U-Net prediction\n{coverage*100:.1f}% intact", fontsize=11)
for ax in axes:
    ax.axis("off")
plt.suptitle(f"mosaic_{IDX+1:04d} — Full pipeline (TEST SET sample near 40%) — BASIC MODEL", fontsize=13)
plt.tight_layout()
plt.show()

print("=" * 60)
print("INFERENCE SUMMARY — BASIC MODEL (TEST SET SAMPLE)")
print("=" * 60)
print(f"Batch:                mosaic_{IDX+1:04d}")
print(f"Actual coverage:      {actual_coverage*100:.1f}%")
print(f"Actual decision:      {actual_decision}")
print("-" * 60)
print(f"U-Net coverage:       {coverage*100:.1f}%")
print(f"Threshold decision:   {threshold_decision}  (>40% → APPROVE)")
print(f"FNN decision:         {fnn_decision}  (prob={fnn_prob:.3f})")
print(f"CNN decision:         {cnn_decision}  (prob={cnn_prob:.3f})")
print("=" * 60)

---
## 8 🏆 Challenge: Beat the Baseline!

The results from the previous sections show there is significant room for improvement. Your task is to redesign the three networks to achieve substantially better performance.

### Your Task

Improve the three networks — **U-Net**, **FNN classifier**, and **CNN classifier** — to achieve the following target accuracy on the test set:

| Method | Target Accuracy |
|---|---|
| Threshold (40% rule) | **≥ 95%** |
| FNN classifier | **≥ 93%** |
| CNN classifier | **≥ 95%** |

### What you can change

You have full freedom to modify the **model architectures** and **hyperparameters**. Think about what could be limiting the current models and what changes might help. Consider:

- Network depth and width
- Types of layers and their configuration
- Training procedure (optimiser, learning rate, number of epochs)
- Loss functions
- Any architectural patterns from the literature that could help

### What is already provided

You do **not** need to rewrite the training loops or evaluation code. The following are provided:
- `train_unet_improved()` — trains the U-Net with validation tracking (loss curves)
- `train_classifier_improved()` — trains either classifier with validation tracking (loss + accuracy curves)
- `evaluate_models()` — the same evaluation function from Section 6
- Plotting cells for training curves to monitor your progress

### Rules
1. You must define three new model classes: `ImprovedUNet`, `ImprovedFNN`, `ImprovedCNN`
2. They must expose the same interface: `forward()`, `get_latent()`, `get_latent_map()` for the U-Net; `forward(z)` for classifiers
3. **Important:** Since the classifiers depend on the U-Net's bottleneck features, you must **retrain all three** after improving the U-Net

Good luck! 🚀

### 8.1 Improved Training Loops (provided)

These training functions track **validation metrics** every epoch so you can monitor progress and diagnose overfitting/underfitting as you iterate on your architectures.

In [ ]:
def train_unet_improved(model, dl_train, dl_val, epochs=50, lr=1e-3):
    """
    Train U-Net with validation tracking.
    Returns history dict with train_loss and val_loss per epoch.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    
    bce = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    history = {"train_loss": [], "val_loss": []}
    
    for ep in range(epochs):
        # ── Training ──────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        for img, mask, _label in dl_train:
            img  = img.to(device)
            mask = mask.to(device)
            
            pred = model(img)
            loss = bce(pred, mask) + dice_loss(pred, mask).mean()
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(dl_train)
        
        # ── Validation ────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for img, mask, _label in dl_val:
                img  = img.to(device)
                mask = mask.to(device)
                pred = model(img)
                loss = bce(pred, mask) + dice_loss(pred, mask).mean()
                val_loss += loss.item()
        val_loss /= len(dl_val)
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        
        if ep % 5 == 0 or ep == epochs - 1:
            print(f"[U-Net] Epoch {ep:3d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    return history


def train_classifier_improved(unet, classifier, dl_train, dl_val, mode="fnn", epochs=50, lr=1e-3):
    """
    Train a classifier (FNN or CNN) with validation tracking.
    
    Args:
        mode: "fnn" uses get_latent(), "cnn" uses get_latent_map()
    
    Returns history dict with train_loss, val_loss, and val_acc per epoch.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    unet.eval()
    classifier.to(device)
    
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(classifier.parameters(), lr=lr)
    
    get_features = unet.get_latent if mode == "fnn" else unet.get_latent_map
    
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    
    for ep in range(epochs):
        # ── Training ──────────────────────────────────────────────
        classifier.train()
        train_loss = 0.0
        for imgs, _masks, labels in dl_train:
            imgs   = imgs.to(device)
            labels = labels.float().to(device)
            
            with torch.no_grad():
                z = get_features(imgs)
            
            pred = classifier(z)
            loss = criterion(pred, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        
        train_loss /= len(dl_train)
        
        # ── Validation ────────────────────────────────────────────
        classifier.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for imgs, _masks, labels in dl_val:
                imgs   = imgs.to(device)
                labels = labels.float().to(device)
                
                z = get_features(imgs)
                pred = classifier(z)
                loss = criterion(pred, labels)
                val_loss += loss.item()
                
                predicted = (pred > 0.5).float()
                correct += (predicted == labels).sum().item()
                total += labels.size(0)
        
        val_loss /= len(dl_val)
        val_acc = correct / total * 100
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        if ep % 5 == 0 or ep == epochs - 1:
            print(f"[{mode.upper()}] Epoch {ep:3d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.1f}%")
    
    return history


def plot_unet_history(history, title="U-Net Training"):
    """Plot train vs validation loss for U-Net."""
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(history["train_loss"], label="Train Loss", color="steelblue")
    ax.plot(history["val_loss"], label="Val Loss", color="coral")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss (BCE + Dice)")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_classifier_history(history, title="Classifier Training"):
    """Plot train/val loss and val accuracy for a classifier."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    
    ax1.plot(history["train_loss"], label="Train Loss", color="steelblue")
    ax1.plot(history["val_loss"], label="Val Loss", color="coral")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss (BCE)")
    ax1.set_title(f"{title} — Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(history["val_acc"], label="Val Accuracy", color="seagreen")
    ax2.axhline(y=95, color="red", linestyle="--", alpha=0.5, label="Target (95%)")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy (%)")
    ax2.set_title(f"{title} — Validation Accuracy")
    ax2.set_ylim([50, 102])
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

### 8.2 Improved U-Net

**Solution:** Add back all the components that were removed:
- ✅ **Skip connections** from encoder to decoder
- ✅ **Double convolution** per block (Conv → BN → ReLU → Conv → BN → ReLU)
- ✅ **BatchNorm** for training stability
- ✅ **`base_ch = 32`** for more representational capacity
- ✅ **BCE + Dice loss** for better region-level optimisation
- ✅ **Adam optimiser** with `lr=1e-3` and **50 epochs**

In [ ]:
class ConvBlock(nn.Module):
    """Double convolution block: Conv → BN → ReLU → Conv → BN → ReLU"""
    
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        return self.block(x)


class ImprovedUNet(nn.Module):
    """
    Full U-Net with skip connections, BatchNorm, and double-conv blocks.
    
    Key improvements over SimpleUNet:
    - Skip connections route encoder features to decoder
    - Double Conv → BN → ReLU per block
    - base_ch=32 (4x more channels)
    """
    
    def __init__(self, in_ch=1, base_ch=32):
        super().__init__()
        self.latent_dim = base_ch * 2
        
        # ── Encoder ───────────────────────────────────────────────────
        self.enc1 = ConvBlock(in_ch, base_ch)
        self.down1 = nn.Conv2d(base_ch, base_ch, 4, 2, 1)
        
        self.enc2 = ConvBlock(base_ch, base_ch * 2)
        self.down2 = nn.Conv2d(base_ch * 2, base_ch * 2, 4, 2, 1)
        
        # ── Bottleneck ────────────────────────────────────────────────
        self.bottleneck = ConvBlock(base_ch * 2, base_ch * 2)
        
        # ── Decoder (WITH skip connections) ───────────────────────────
        self.up1 = nn.ConvTranspose2d(base_ch * 2, base_ch * 2, 4, 2, 1)
        self.dec1 = ConvBlock(base_ch * 4, base_ch)   # cat with enc2 skip
        
        self.up2 = nn.ConvTranspose2d(base_ch, base_ch, 4, 2, 1)
        self.dec2 = ConvBlock(base_ch * 2, base_ch)   # cat with enc1 skip
        
        # ── Output head ──────────────────────────────────────────────
        self.out_conv = nn.Conv2d(base_ch, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        d1 = self.down1(e1)

        e2 = self.enc2(d1)
        d2 = self.down2(e2)

        b = self.bottleneck(d2)

        u1 = self.up1(b)
        u1 = torch.cat([u1, e2], dim=1)   # ← skip connection!
        d1 = self.dec1(u1)

        u2 = self.up2(d1)
        u2 = torch.cat([u2, e1], dim=1)   # ← skip connection!
        d2 = self.dec2(u2)

        return self.out_conv(d2)

    def get_latent(self, x):
        """Returns pooled latent vector (B, 64) via Global Average Pooling."""
        e1 = self.enc1(x);  d1 = self.down1(e1)
        e2 = self.enc2(d1); d2 = self.down2(e2)
        b = self.bottleneck(d2)
        return b.mean(dim=(2, 3))

    def get_latent_map(self, x):
        """Returns spatial bottleneck map (B, 64, 64, 64) without pooling."""
        e1 = self.enc1(x);  d1 = self.down1(e1)
        e2 = self.enc2(d1); d2 = self.down2(e2)
        return self.bottleneck(d2)

Instantiate the improved U-Net and inspect its architecture.

In [ ]:
unet_improved = ImprovedUNet(in_ch=1, base_ch=32)
print(unet_improved)
print(f"\nBottleneck channels (latent_dim): {unet_improved.latent_dim}")

Train the improved U-Net with validation tracking. This uses BCE + Dice loss and the Adam optimiser for 50 epochs.

In [ ]:
# Train the improved U-Net with validation tracking
unet_history = train_unet_improved(unet_improved, dl_train, dl_val, epochs=50, lr=1e-3)

Plot the training and validation loss curves to check for convergence and overfitting.

In [ ]:
plot_unet_history(unet_history, title="Improved U-Net — Train vs Val Loss")

Visualise the improved U-Net's segmentation predictions on the test set — compare against the basic model's output from Section 3.

In [ ]:
# Visualize improved U-Net predictions on TEST SET
unet_improved.eval()
unet_improved_device = next(unet_improved.parameters()).device

test_sample_indices = list(ds_test.indices[:8])

fig, axes = plt.subplots(3, 8, figsize=(18, 9))
for col, idx in enumerate(test_sample_indices):
    img, mask, label = ds_full[idx]
    img_t = img.unsqueeze(0).to(unet_improved_device)

    with torch.no_grad():
        pred = torch.sigmoid(unet_improved(img_t)).squeeze().cpu()
    pred_bin = (pred > 0.5).float()
    
    status = "APPROVE" if label == 1 else "REJECT"
    coverage = ground_truth.iloc[idx]["coverage_pct"] * 100

    axes[0, col].imshow(img.squeeze(), cmap="gray")
    axes[0, col].set_title(f"mosaic_{idx+1:04d}\n{status}", fontsize=8)
    axes[0, col].axis("off")

    axes[1, col].imshow(mask.squeeze(), cmap="gray")
    axes[1, col].set_title(f"GT ({coverage:.1f}%)", fontsize=8)
    axes[1, col].axis("off")

    axes[2, col].imshow(pred_bin, cmap="gray")
    axes[2, col].set_title("U-Net pred", fontsize=8)
    axes[2, col].axis("off")

for row, lbl in enumerate(["Original", "GT mask", "U-Net pred"]):
    axes[row, 0].set_ylabel(lbl, fontsize=11)

plt.suptitle("U-Net segmentation results on TEST SET (unseen data) — IMPROVED MODEL", fontsize=13)
plt.tight_layout()
plt.subplots_adjust(hspace=0.3) 
plt.show()

### 8.3 Improved FNN Classifier

**Solution:** Add a hidden layer with ReLU activation:
- ✅ **Hidden layer** `Linear(64 → 64) + ReLU` before the output
- ✅ **Adam optimiser** with `lr=1e-3`
- ✅ **50 training epochs**

In [ ]:
class ImprovedFNN(nn.Module):
    """
    Improved feedforward classifier with a hidden layer.
    
    Key improvements over SimpleApprovalFNN:
    - Added hidden layer Linear(64 → 64) + ReLU
    - More capacity for non-linear decision boundaries
    """
    
    def __init__(self, latent_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, z):
        return self.net(z).squeeze(-1)

Instantiate and train the improved FNN classifier on the improved U-Net's latent vectors.

In [ ]:
classifier_fnn_improved = ImprovedFNN(latent_dim=unet_improved.latent_dim)
fnn_history = train_classifier_improved(
    unet_improved, classifier_fnn_improved, dl_train, dl_val,
    mode="fnn", epochs=50, lr=1e-3
)

Plot the FNN training curves to verify convergence.

In [ ]:
plot_classifier_history(fnn_history, title="Improved FNN Classifier")

### 8.4 Improved CNN Classifier

**Solution:** Add a second conv layer, BatchNorm, and MaxPool:
- ✅ **Two conv layers** with BatchNorm and ReLU
- ✅ **MaxPool2d(2)** between layers for wider receptive field
- ✅ **More filters** (32 → 16) for richer feature extraction
- ✅ **Adam optimiser** with `lr=1e-3`
- ✅ **50 training epochs**

In [ ]:
class ImprovedCNN(nn.Module):
    """
    Improved convolutional classifier with BatchNorm, MaxPool, and two conv layers.
    
    Key improvements over SimpleApprovalCNN:
    - Two conv layers instead of one
    - BatchNorm after each conv
    - MaxPool2d(2) between conv layers
    - More channels (32 → 16 instead of just 4)
    """
    
    def __init__(self, in_channels=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )
    
    def forward(self, z_map):
        x = self.features(z_map)
        return self.classifier(x).squeeze(-1)

Instantiate and train the improved CNN classifier on the improved U-Net's spatial latent maps.

In [ ]:
classifier_cnn_improved = ImprovedCNN(in_channels=unet_improved.latent_dim)
cnn_history = train_classifier_improved(
    unet_improved, classifier_cnn_improved, dl_train, dl_val,
    mode="cnn", epochs=50, lr=1e-3
)

Plot the CNN training curves to verify convergence.

In [ ]:
plot_classifier_history(cnn_history, title="Improved CNN Classifier")

### 8.5 Final Evaluation — Improved Models

In [ ]:
df_improved, acc_improved = evaluate_models(
    unet_improved, classifier_fnn_improved, classifier_cnn_improved,
    ds_full, ds_test, ground_truth,
    model_label="IMPROVED MODELS"
)

print("\nFirst 20 predictions:")
df_improved.head(20).set_index("Image")

Plot the accuracy bar chart for the improved models.

In [ ]:
plot_accuracy_comparison(acc_improved, title="Accuracy Comparison — Improved Models")

### 8.6 Side-by-Side Comparison: Basic vs Improved

In [ ]:
methods = ["Threshold", "FNN", "CNN"]

x = np.arange(len(methods))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, acc_basic, width, label="Basic Models", color=["#D4A843", "#D47070", "#70A870"], alpha=0.6, edgecolor="gray")
bars2 = ax.bar(x + width/2, acc_improved, width, label="Improved Models", color=["goldenrod", "tomato", "seagreen"], edgecolor="gray")

ax.set_ylabel("Accuracy (%)")
ax.set_title("Basic vs Improved Models — Test Set Accuracy")
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylim([0, 105])
ax.legend()
ax.axhline(y=95, color="red", linestyle="--", alpha=0.4, label="Target (95%)")

for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=9, color="gray")

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{height:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n{'='*65}")
print(f"  IMPROVEMENT SUMMARY")
print(f"{'='*65}")
for m, b, i in zip(methods, acc_basic, acc_improved):
    delta = i - b
    print(f"  {m:12s}: {b:5.1f}% → {i:5.1f}%  ({'+' if delta >= 0 else ''}{delta:.1f}% change)")
print(f"{'='*65}")

### 8.7 Improved Pipeline Inference on a Single Image

In [ ]:
# Reuse the same threshold sample from Section 7
img, gt_mask, label = ds_full[IDX]
img_t = img.unsqueeze(0).to(unet_improved_device)

unet_improved.eval()
classifier_fnn_improved.eval()
classifier_cnn_improved.eval()

with torch.no_grad():
    logits = unet_improved(img_t)
    seg_pred = torch.sigmoid(logits)
    seg_bin = (seg_pred > 0.5).float()
    coverage = seg_pred.mean().item()
    threshold_decision = "APPROVE" if coverage > 0.4 else "REJECT"

    z = unet_improved.get_latent(img_t)
    fnn_prob = classifier_fnn_improved(z).item()
    fnn_decision = "APPROVE" if fnn_prob > 0.5 else "REJECT"

    z_map = unet_improved.get_latent_map(img_t)
    cnn_prob = classifier_cnn_improved(z_map).item()
    cnn_decision = "APPROVE" if cnn_prob > 0.5 else "REJECT"

actual_decision = "APPROVE" if label == 1 else "REJECT"
actual_coverage = ground_truth.iloc[IDX]["coverage_pct"]

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(img.squeeze(), cmap="gray")
axes[0].set_title(f"Original: mosaic_{IDX+1:04d}", fontsize=11)
axes[1].imshow(gt_mask.squeeze(), cmap="gray")
axes[1].set_title(f"Ground truth mask\n{actual_coverage*100:.1f}% intact", fontsize=11)
axes[2].imshow(seg_bin.squeeze().cpu(), cmap="gray")
axes[2].set_title(f"U-Net prediction\n{coverage*100:.1f}% intact", fontsize=11)
for ax in axes:
    ax.axis("off")
plt.suptitle(f"mosaic_{IDX+1:04d} — Full pipeline (TEST SET sample near 40%) — IMPROVED MODEL", fontsize=13)
plt.tight_layout()
plt.show()

print("=" * 60)
print("INFERENCE SUMMARY — IMPROVED MODEL (TEST SET SAMPLE)")
print("=" * 60)
print(f"Batch:                mosaic_{IDX+1:04d}")
print(f"Actual coverage:      {actual_coverage*100:.1f}%")
print(f"Actual decision:      {actual_decision}")
print("-" * 60)
print(f"U-Net coverage:       {coverage*100:.1f}%")
print(f"Threshold decision:   {threshold_decision}  (>40% → APPROVE)")
print(f"FNN decision:         {fnn_decision}  (prob={fnn_prob:.3f})")
print(f"CNN decision:         {cnn_decision}  (prob={cnn_prob:.3f})")
print("=" * 60)

#### Takeaways

**What the improvements demonstrate:**

| Change | Effect |
|---|---|
| **Skip connections** | Sharp, pixel-accurate segmentation masks instead of blurry blobs |
| **BatchNorm** | Stable training, faster convergence, higher effective learning rate |
| **Double conv blocks** | Richer feature extraction at each encoder/decoder stage |
| **More channels** (8→32) | Higher representational capacity for the bottleneck |
| **Dice loss** | Direct optimisation for region overlap (not just per-pixel accuracy) |
| **Adam optimiser** | Adaptive learning rates, much faster convergence than SGD |
| **Hidden layer (FNN)** | Non-linear decision boundary instead of logistic regression |
| **Two conv layers + BN + MaxPool (CNN)** | Spatial feature hierarchy before aggregation |
| **More epochs** (10→50) | Sufficient training time for the larger models to converge |

**Three classification strategies, three philosophies:**

- **Threshold (40% rule)** is a **deterministic** approach — purely rule-based. It counts white pixels in the U-Net mask and applies the 40% cutoff. Fast and interpretable, but completely dependent on segmentation quality.
- The **FNN classifier** collapses the entire bottleneck into a 64-number summary via Global Average Pooling, then runs a two-layer MLP. It can learn to compensate for systematic U-Net errors, but loses spatial information.
- The **CNN classifier** processes the bottleneck as a spatial feature map, using convolutional layers before aggregating. It can detect spatial patterns (e.g., "large contiguous intact area" vs "scattered small patches") before collapsing to a decision.

> **Key insight:** When the task depends on spatial structure — as surface area quality assessment does — preserving spatial structure through convolutional processing before aggregation is the better design.

**Business impact:**
- **False positives** (approving bad batches) → unhappy customers, returns
- **False negatives** (rejecting good batches) → wasted manual inspection time, lost sales

The CNN classifier's ability to understand spatial coverage patterns makes it the strongest candidate for production deployment.